In [1]:
import os
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILE_DISABLE"] = "1"

os.chdir(r"C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL")
print("当前工作目录:", os.getcwd())

当前工作目录: C:\Users\14731\desktop\Capstone\Wifi-indoor-localization-with-rtt-and-csi\DL


In [2]:
import pandas as pd
import numpy as np
import ast

def _parse_csi_cell(x):
    if isinstance(x, str):
        return np.array(ast.literal_eval(x), dtype=np.float32)
    return np.array(x, dtype=np.float32)

def align_ftm_anchor(
    ftm_csv: str,
    csi_csv: str,
    window_ms: int = 300,
    aggregate: str = "median",   # "median" | "mean" | "nearest" | "trimmed_mean"
    min_csi_in_window: int = 2,  # 窗内至少几条 CSI 才产出样本
    trim_ratio: float = 0.2,     # trimmed_mean 的裁剪比例（左右各 trim_ratio）
    return_debug: bool = False   # 是否返回一些诊断信息
):
    """
    以每条 FTM 为锚点，在其时间窗 [t-window, t+window] 内聚合多条 CSI → 1 个样本。
    返回:
        csi_list:  每个样本一条 128 维 I/Q 交错数组（float32，已聚合）
        rssi_list: 聚合后的 RSSI（float）
        rtt_list:  该 FTM 的 RTT（ns）
        ts_list:   FTM 时间戳（pd.Timestamp）
        (可选) dbg: 诊断字典，含每个样本的匹配 CSI 数、最近Δt等
    """
    # 读取
    ftm_df = pd.read_csv(ftm_csv)
    csi_df = pd.read_csv(csi_csv)

    # 列名 & 时间
    ftm_df.rename(columns=lambda x: x.strip().lower(), inplace=True)
    csi_df.rename(columns=lambda x: x.strip().lower(), inplace=True)
    ftm_df['timestamp'] = pd.to_datetime(ftm_df['timestamp'], errors='coerce')
    csi_df['timestamp'] = pd.to_datetime(csi_df['timestamp'], errors='coerce')
    ftm_df.dropna(subset=['timestamp'], inplace=True)
    csi_df.dropna(subset=['timestamp'], inplace=True)
    ftm_df.sort_values('timestamp', inplace=True)
    csi_df.sort_values('timestamp', inplace=True)

    window = pd.Timedelta(milliseconds=window_ms)

    out_csi, out_rssi, out_rtt, out_ts = [], [], [], []
    dbg_counts, dbg_nearest_dt_ms = [], []

    for _, ftm_row in ftm_df.iterrows():
        t = ftm_row['timestamp']
        mask = (csi_df['timestamp'] >= t - window) & (csi_df['timestamp'] <= t + window)
        csi_win = csi_df[mask]
        n = len(csi_win)
        if n < min_csi_in_window:
            continue

        # 最近的一条索引（用于 nearest 或调试）
        nearest_idx = (np.abs(csi_win['timestamp'] - t)).idxmin()
        dt_nearest_ms = float(np.abs(csi_win.loc[nearest_idx, 'timestamp'] - t) / pd.Timedelta(milliseconds=1))

        # 解析窗口内所有 CSI 与 RSSI
        csi_arrs = [_parse_csi_cell(r['data']) for _, r in csi_win.iterrows()]
        rssi_vals = [float(r['rssi']) for _, r in csi_win.iterrows()]
        csi_stack = np.stack(csi_arrs, axis=0).astype(np.float32)  # [n, 128]
        rssi_arr  = np.asarray(rssi_vals, dtype=np.float32)

        # 聚合
        if aggregate == "median":
            csi_agg = np.median(csi_stack, axis=0).astype(np.float32)
            rssi_agg = float(np.median(rssi_arr))
        elif aggregate == "mean":
            csi_agg = np.mean(csi_stack, axis=0).astype(np.float32)
            rssi_agg = float(np.mean(rssi_arr))
        elif aggregate == "nearest":
            row = csi_win.loc[nearest_idx]
            csi_agg = _parse_csi_cell(row['data']).astype(np.float32)
            rssi_agg = float(row['rssi'])
        elif aggregate == "trimmed_mean":
            # 按时间靠近程度排序后做截尾均值
            order = np.argsort(np.abs(csi_win['timestamp'].to_numpy() - t))
            csi_sorted = csi_stack[order]
            rssi_sorted = rssi_arr[order]
            k = int(np.floor(trim_ratio * n))
            sl = slice(k, n - k if n - k > k else n)  # 避免空切片
            csi_agg = np.mean(csi_sorted[sl], axis=0).astype(np.float32)
            rssi_agg = float(np.mean(rssi_sorted[sl]))
        else:
            raise ValueError("aggregate must be one of {'median','mean','nearest','trimmed_mean'}")

        out_csi.append(csi_agg)
        out_rssi.append(rssi_agg)
        out_rtt.append(float(ftm_row.get('rtt_raw (nsec)', ftm_row.get('rtt_ns', np.nan))))
        out_ts.append(t)

        dbg_counts.append(int(n))
        dbg_nearest_dt_ms.append(dt_nearest_ms)

    if return_debug:
        dbg = {
            "num_csi_in_window": np.array(dbg_counts, dtype=np.int32),
            "nearest_dt_ms": np.array(dbg_nearest_dt_ms, dtype=np.float32),
            "window_ms": window_ms,
            "aggregate": aggregate,
        }
        return out_csi, out_rssi, out_rtt, out_ts, dbg

    return out_csi, out_rssi, out_rtt, out_ts


csi_list, rssi_list, rtt_list, ts_list, dbg = align_ftm_anchor(
    ftm_csv="data/ftm_data_0.3m.csv",
    csi_csv="data/csi_data_0.3m.csv",
    window_ms=300,          # 300–500ms 常见
    aggregate="median",     # 稳健
    min_csi_in_window=2,
    return_debug=True
)

print(f"样本数: {len(csi_list)}")
print("每窗匹配 CSI 数量分布:", np.bincount(dbg["num_csi_in_window"]))
print("最近CSI与FTM的Δt(ms)：均值/中位/95分位 =",
      np.mean(dbg["nearest_dt_ms"]), np.median(dbg["nearest_dt_ms"]),
      np.percentile(dbg["nearest_dt_ms"], 95))

样本数: 230
每窗匹配 CSI 数量分布: [  0   0   8 216   6]
最近CSI与FTM的Δt(ms)：均值/中位/95分位 = 41.36878 36.276 66.33604831695546


In [9]:
import os
import glob
import re
import numpy as np
import random

# 确保你已引入 align_ftm_anchor
# from your_module import align_ftm_anchor

def load_all_data(
    data_dir,
    window_ms=300,             # 推荐 300–500ms
    aggregate="median",        # "median" | "mean" | "nearest" | "trimmed_mean"
    min_csi_in_window=2,       # 每个 FTM 窗至少要有几条 CSI 才产出样本
    trim_ratio=0.2,            # 截尾均值的比例
    shuffle=False,              # 是否打乱
    verbose=True               # 打印诊断信息
):
    """
    扫描 data_dir 下的 csi_data_*.csv 和 ftm_data_*.csv，按 FTM 为锚点对齐聚合。
    返回:
        csi_list_all:  List[np.ndarray], 每项是 128 维 I/Q 交错（已聚合）
        rssi_list_all: List[float]
        rtt_list_all:  List[float] (ns)
        dist_list_all: List[float]，与样本一一对应的标称距离
    """
    csi_list_all, rssi_list_all, rtt_list_all, dist_list_all = [], [], [], []

    # 找到所有 csi 文件
    csi_files = glob.glob(os.path.join(data_dir, "csi_data_*.csv"))
    csi_files.sort()  # 稳定顺序（可按需要改为自然排序）

    total_before_shuffle = 0

    for csi_file in csi_files:
        # 提取距离 (比如 5.7 from csi_data_5.7m.csv)
        m = re.search(r"csi_data_(\d+(?:\.\d+)?)m\.csv", os.path.basename(csi_file))
        if not m:
            if verbose:
                print(f"跳过无法解析距离的文件: {os.path.basename(csi_file)}")
            continue
        dist = float(m.group(1))

        # 找对应的 ftm 文件
        ftm_file = os.path.join(data_dir, f"ftm_data_{dist}m.csv")
        if not os.path.exists(ftm_file):
            if verbose:
                print(f"⚠️ 找不到 {os.path.basename(ftm_file)}，跳过")
            continue

        # 以 FTM 为锚点对齐并聚合
        try:
            csi_list, rssi_list, rtt_list, ts_list, dbg = align_ftm_anchor(
                ftm_csv=ftm_file,
                csi_csv=csi_file,
                window_ms=window_ms,
                aggregate=aggregate,
                min_csi_in_window=min_csi_in_window,
                trim_ratio=trim_ratio,
                return_debug=True
            )
        except Exception as e:
            if verbose:
                print(f"❌ 对齐失败 {os.path.basename(csi_file)} / {os.path.basename(ftm_file)}: {e}")
            continue

        n = len(csi_list)
        if n == 0:
            if verbose:
                print(f"⚠️ 距离 {dist} m 无有效样本（可能窗口太小或时间戳不匹配）")
            continue

        # 累加样本与标签（用文件名里的标称距离作为标签）
        csi_list_all.extend(csi_list)
        rssi_list_all.extend(rssi_list)
        rtt_list_all.extend(rtt_list)
        dist_list_all.extend([dist] * n)
        total_before_shuffle += n

        if verbose:
            counts = np.bincount(dbg["num_csi_in_window"])
            mean_dt = float(np.mean(dbg["nearest_dt_ms"])) if len(dbg["nearest_dt_ms"]) else float("nan")
            med_dt  = float(np.median(dbg["nearest_dt_ms"])) if len(dbg["nearest_dt_ms"]) else float("nan")
            p95_dt  = float(np.percentile(dbg["nearest_dt_ms"], 95)) if len(dbg["nearest_dt_ms"]) else float("nan")
            print(f"[{dist:.2f} m] 样本数: {n} | 每窗CSI数分布: {counts.tolist()} | "
                  f"Δt最近(ms): 均值 {mean_dt:.2f}, 中位 {med_dt:.2f}, 95分位 {p95_dt:.2f}")

    if total_before_shuffle == 0:
        if verbose:
            print("❗ 没有载入到任何样本，请检查文件命名与时间戳。")
        return [], [], [], []

    # 可选：打乱
    if shuffle:
        idx = list(range(total_before_shuffle))
        random.shuffle(idx)
        csi_list_all = [csi_list_all[i] for i in idx]
        rssi_list_all = [rssi_list_all[i] for i in idx]
        rtt_list_all = [rtt_list_all[i] for i in idx]
        dist_list_all = [dist_list_all[i] for i in idx]

    if verbose:
        print(f"✅ 总样本数: {len(csi_list_all)}（聚合方式: {aggregate}, 窗口: ±{window_ms}ms）")

    return csi_list_all, rssi_list_all, rtt_list_all, dist_list_all

csi_list, rssi_list, rtt_list, dist_list = load_all_data("data")
csi_list_test, rssi_list_test, rtt_list_test, dist_list_test = load_all_data("data/test_data")

[0.30 m] 样本数: 230 | 每窗CSI数分布: [0, 0, 8, 216, 6] | Δt最近(ms): 均值 41.37, 中位 36.28, 95分位 66.34
[0.60 m] 样本数: 207 | 每窗CSI数分布: [0, 0, 7, 153, 29, 10, 6, 1, 1] | Δt最近(ms): 均值 49.08, 中位 46.47, 95分位 72.81
[0.90 m] 样本数: 210 | 每窗CSI数分布: [0, 0, 5, 201, 4] | Δt最近(ms): 均值 56.69, 中位 54.08, 95分位 68.02
[1.20 m] 样本数: 195 | 每窗CSI数分布: [0, 0, 5, 177, 12, 1] | Δt最近(ms): 均值 62.01, 中位 60.41, 95分位 78.15
[1.50 m] 样本数: 222 | 每窗CSI数分布: [0, 0, 8, 192, 21, 0, 1] | Δt最近(ms): 均值 71.87, 中位 70.91, 95分位 89.00
[1.80 m] 样本数: 199 | 每窗CSI数分布: [0, 0, 12, 176, 10, 0, 1] | Δt最近(ms): 均值 75.40, 中位 79.51, 95分位 93.21
[2.10 m] 样本数: 195 | 每窗CSI数分布: [0, 0, 28, 158, 9] | Δt最近(ms): 均值 89.98, 中位 88.66, 95分位 102.27
[2.40 m] 样本数: 209 | 每窗CSI数分布: [0, 0, 49, 138, 22] | Δt最近(ms): 均值 91.24, 中位 94.96, 95分位 105.31
[2.70 m] 样本数: 208 | 每窗CSI数分布: [0, 0, 51, 140, 16, 1] | Δt最近(ms): 均值 88.18, 中位 94.29, 95分位 102.44
[3.00 m] 样本数: 204 | 每窗CSI数分布: [0, 0, 7, 189, 7, 1] | Δt最近(ms): 均值 84.17, 中位 87.14, 95分位 96.28
[3.30 m] 样本数: 211 | 每窗CSI数分布: [0, 0, 4, 196

In [10]:
def build_and_save_npz(
    csi_list: List[List[float]],
    rssi_list: List[float],
    rtt_list: List[float],
    dist_list: List[float],
    out_path: str = "dataset_train_ready.npz",
    window_T: int = 30,
    do_zscore: bool = True,
    min_valid_frames_per_window: int = None,
    apply_stats: Dict[str, np.ndarray] | None = None,  # ← 新增：传入已有 mu/sigma
) -> Tuple[np.ndarray, np.ndarray, Dict[str, np.ndarray] | None]:
    """
    如果 apply_stats=None 且 do_zscore=True：对当前 X 拟合 z-score 并保存 mu/sigma（训练集用）
    如果 apply_stats 提供：用它来做 z-score，不再拟合（测试集用）
    """
    N = len(csi_list)
    assert N == len(rssi_list) == len(rtt_list) == len(dist_list), "四个列表长度必须一致"
    if min_valid_frames_per_window is None:
        # 单窗场景放宽门槛
        min_valid_frames_per_window = 1 if window_T <= 3 else max(5, int(np.ceil(window_T * 0.3)))

    X_rows, y_rows = [], []

    for start in range(0, N - window_T + 1, window_T):
        end = start + window_T

        chunk_csi  = csi_list[start:end]
        chunk_rssi = np.asarray(rssi_list[start:end], dtype=np.float32)
        chunk_rtt  = np.asarray(rtt_list[start:end],  dtype=np.float64)  # ns
        chunk_dist = np.asarray(dist_list[start:end], dtype=np.float32)

        amp_list, dphi_list = [], []
        for raw in chunk_csi:
            try:
                amp, dphi = _csi_frame_to_amp_dphi(raw)
                amp_list.append(amp); dphi_list.append(dphi)
            except Exception:
                continue

        if len(amp_list) < min_valid_frames_per_window:
            continue

        amp_arr  = np.stack(amp_list, axis=0)   # [t, 52]
        dphi_arr = np.stack(dphi_list, axis=0)  # [t, 51]

        amp_med  = np.median(amp_arr, axis=0)
        amp_iqr  = _iqr(amp_arr, axis=0)
        dphi_med = np.median(dphi_arr, axis=0)
        dphi_iqr = _iqr(dphi_arr, axis=0)
        feat_csi = np.concatenate([amp_med, amp_iqr, dphi_med, dphi_iqr], axis=0)  # 206

        rtt_min = float(np.min(chunk_rtt))
        rtt_med = float(np.median(chunk_rtt))
        rtt_mean = float(np.mean(chunk_rtt))
        rtt_std = float(np.std(chunk_rtt))
        rtt_iqr = float(_iqr(chunk_rtt))
        rtt_p05 = float(np.percentile(chunk_rtt, 5))
        rtt_p95 = float(np.percentile(chunk_rtt, 95))
        rtt_min_to_dist = (rtt_min * 1e-9) * C / 2.0

        feat_rtt = np.array(
            [rtt_min, rtt_med, rtt_mean, rtt_std, rtt_iqr, rtt_p05, rtt_p95, rtt_min_to_dist],
            dtype=np.float32
        )

        rssi_med = float(np.median(chunk_rssi))
        rssi_iqr = float(_iqr(chunk_rssi))
        rssi_mean = float(np.mean(chunk_rssi))
        rssi_std = float(np.std(chunk_rssi))
        rssi_min = float(np.min(chunk_rssi))
        rssi_max = float(np.max(chunk_rssi))
        feat_rssi = np.array([rssi_med, rssi_iqr, rssi_mean, rssi_std, rssi_min, rssi_max], dtype=np.float32)

        feat = np.concatenate([feat_csi.astype(np.float32), feat_rtt, feat_rssi], axis=0)
        X_rows.append(feat)
        y_rows.append(float(np.median(chunk_dist)))

    if not X_rows:
        raise ValueError("没有产生任何窗口样本。请检查 window_T、数据对齐或 CSI 解析。")

    X = np.stack(X_rows, axis=0).astype(np.float32)
    y = np.asarray(y_rows, dtype=np.float32)

    stats = None
    if do_zscore:
        if apply_stats is not None:
            # 用外部（训练集）的 mu/sigma 标准化（测试集使用）
            X = _zscore_transform(X, apply_stats)
        else:
            # 拟合并返回（训练集使用）
            stats = _zscore_fit(X)
            X = _zscore_transform(X, stats)

    # 保存（若提供 apply_stats，我们不重复保存 mu/sigma）
    if stats is not None:
        np.savez(out_path, X=X, y=y, mu=stats["mu"], sigma=stats["sigma"])
    else:
        np.savez(out_path, X=X, y=y)

    return X, y, stats

# 1) 训练集：拟合并保存 mu/sigma
X_tr, y_tr, stats_tr = build_and_save_npz(
    csi_list, rssi_list, rtt_list, dist_list,
    out_path="dataset_ftm_train.npz",
    window_T=1,
    do_zscore=True,
    min_valid_frames_per_window=1,
    apply_stats=None,               # 训练集不传
)

# 2) 测试集：复用训练集的 mu/sigma（不要再拟合）
X_te, y_te, _ = build_and_save_npz(
    csi_list_test, rssi_list_test, rtt_list_test, dist_list_test,
    out_path="dataset_ftm_test.npz",
    window_T=1,
    do_zscore=True,
    min_valid_frames_per_window=1,
    apply_stats=stats_tr,           # ← 关键：用训练集的统计量
)

In [11]:
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

import torch
import torch.version as _tv  # 强制导入 torch.version 子模块
torch.version = _tv 

# 1) 读数据
data = np.load("dataset_ftm_train.npz")   # 你保存的 npz
X = data["X"].astype(np.float32)
y = data["y"].astype(np.float32)

# 2) 按“距离点位”分组留出测试集（避免同点位泄露）
#    假设你是每 0.3 m 采集一次
STEP = 0.3
labels = np.round(y / STEP) * STEP                      # 归并到0.3m网格
uniq = np.array(sorted(list(set(labels.tolist()))))
# 留出每隔 4 个点位作为测试（你也可自选一段，比如 3.0–4.5m）
test_points = set(uniq[::4])
test_mask = np.array([l in test_points for l in labels])
train_mask = ~test_mask

Xtr, ytr = X[train_mask], y[train_mask]
Xte, yte = X[test_mask], y[test_mask]

# 3) DataLoader
bs = 128
tr_loader = DataLoader(TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr)), batch_size=bs, shuffle=True)
te_loader = DataLoader(TensorDataset(torch.from_numpy(Xte), torch.from_numpy(yte)), batch_size=bs, shuffle=False)

# 4) 模型（简单 MLP）
class MLP(nn.Module):
    def __init__(self, in_dim=220, hidden=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.LayerNorm(hidden),
            nn.Linear(hidden, hidden//2), nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden//2, 1)
        )
    def forward(self, x): return self.net(x).squeeze(1)

model = MLP(220, 256)
opt = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
loss_fn = nn.HuberLoss(delta=0.1)  # 对离群点更鲁棒

# 5) 训练 + 评估
def evaluate(dl):
    model.eval()
    mae, mse, n = 0.0, 0.0, 0
    with torch.no_grad():
        for xb, yb in dl:
            pred = model(xb)
            err = (pred - yb).cpu().numpy()
            mae += np.abs(err).sum()
            mse += (err**2).sum()
            n += yb.numel()
    return mae/n, np.sqrt(mse/n)

for epoch in range(30):
    model.train()
    for xb, yb in tr_loader:
        opt.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward(); opt.step()
    tr_mae, tr_rmse = evaluate(tr_loader)
    te_mae, te_rmse = evaluate(te_loader)
    print(f"ep{epoch+1:02d} | train MAE {tr_mae:.03f}m RMSE {tr_rmse:.03f}m | test MAE {te_mae:.03f}m RMSE {te_rmse:.03f}m")

# 6) 推理函数（记得保持与训练相同的 z-score；已在 npz 中用 X 保存为标准化后的）
def predict(X_new: np.ndarray) -> np.ndarray:
    with torch.no_grad():
        return model(torch.from_numpy(X_new.astype(np.float32))).cpu().numpy()


ep01 | train MAE 0.391m RMSE 0.532m | test MAE 0.661m RMSE 0.798m
ep02 | train MAE 0.324m RMSE 0.447m | test MAE 0.605m RMSE 0.739m
ep03 | train MAE 0.261m RMSE 0.362m | test MAE 0.503m RMSE 0.626m
ep04 | train MAE 0.227m RMSE 0.311m | test MAE 0.463m RMSE 0.547m
ep05 | train MAE 0.233m RMSE 0.315m | test MAE 0.491m RMSE 0.610m
ep06 | train MAE 0.211m RMSE 0.292m | test MAE 0.489m RMSE 0.582m
ep07 | train MAE 0.190m RMSE 0.255m | test MAE 0.452m RMSE 0.528m
ep08 | train MAE 0.231m RMSE 0.302m | test MAE 0.407m RMSE 0.524m
ep09 | train MAE 0.192m RMSE 0.260m | test MAE 0.406m RMSE 0.493m
ep10 | train MAE 0.201m RMSE 0.261m | test MAE 0.504m RMSE 0.574m
ep11 | train MAE 0.202m RMSE 0.263m | test MAE 0.439m RMSE 0.552m
ep12 | train MAE 0.182m RMSE 0.239m | test MAE 0.499m RMSE 0.590m
ep13 | train MAE 0.227m RMSE 0.309m | test MAE 0.430m RMSE 0.537m
ep14 | train MAE 0.166m RMSE 0.222m | test MAE 0.449m RMSE 0.523m
ep15 | train MAE 0.172m RMSE 0.225m | test MAE 0.453m RMSE 0.529m
ep16 | tra

In [13]:
import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset

# ===== 1) 用训练时的 stats_tr，把测试集做成 220 维特征并做同一套 z-score =====
# 你已经有：csi_list_test, rssi_list_test, rtt_list_test, dist_list_test
# 并且已有函数 build_and_save_npz / _csi_frame_to_amp_dphi 等

# 如果不想在磁盘上留下文件，随便给个临时路径，生成后不去读它即可
X_te, y_te, _ = build_and_save_npz(
    csi_list_test, rssi_list_test, rtt_list_test, dist_list_test,
    out_path="_tmp_ignore_test.npz",  # 只是占位，可随便写
    window_T=1,
    do_zscore=True,
    min_valid_frames_per_window=1,
    apply_stats=stats_tr              # 关键：用训练集的 mu/sigma 做归一化
)

# 若你**没有** stats_tr，但手里还留着训练时用的 X_tr，可以临时凑一个：
# stats_tr = {"mu": X_tr.mean(axis=0, keepdims=True), "sigma": X_tr.std(axis=0, keepdims=True)+1e-8}
# 然后手动：X_te = ((X_te_raw - stats_tr["mu"]) / stats_tr["sigma"]).astype(np.float32)

# ===== 2) 用内存里的 model 直接推理 =====
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

dl = DataLoader(TensorDataset(torch.from_numpy(X_te), torch.from_numpy(y_te)),
                batch_size=512, shuffle=False)

preds, trues = [], []
with torch.no_grad():
    for xb, yb in dl:
        xb = xb.to(device)
        pb = model(xb).cpu().numpy()
        preds.append(pb); trues.append(yb.numpy())

pred = np.concatenate(preds, axis=0)
y    = np.concatenate(trues, axis=0)

# ===== 3) 指标（含 ±0.3 m 判正确）=====
THRESH = 0.3
bins = np.array([2.5, 5.0, 6.0])

ae   = np.abs(pred - y)
mae  = float(np.mean(ae))
rmse = float(np.sqrt(np.mean((pred - y) ** 2)))
med  = float(np.median(ae))
p95  = float(np.percentile(ae, 95))
acc  = float(np.mean(ae <= THRESH))
print(f"Overall | MAE {mae:.3f} m | RMSE {rmse:.3f} m | MedianAE {med:.3f} m | P95AE {p95:.3f} m | Acc@{THRESH:.1f}m {acc*100:.1f}%")

# 分距离桶
labels = bins[np.argmin(np.abs(y[:, None] - bins[None, :]), axis=1)]
uniq = np.array(sorted(list(set(labels.tolist()))))
print("\nPer-distance bucket (rounded to {:.1f} m):".format(STEP))
print("dist   N    Acc@{:.1f}m   MAE    MedAE   P95AE".format(THRESH))
for d in uniq:
    m = labels == d
    if not np.any(m): continue
    ae_d = ae[m]
    acc_d = float(np.mean(ae_d <= THRESH))
    mae_d = float(np.mean(ae_d))
    med_d = float(np.median(ae_d))
    p95_d = float(np.percentile(ae_d, 95))
    print(f"{d:4.1f}  {m.sum():4d}   {acc_d*100:6.1f}%   {mae_d:5.3f}  {med_d:5.3f}  {p95_d:5.3f}")


Overall | MAE 0.382 m | RMSE 0.503 m | MedianAE 0.298 m | P95AE 1.050 m | Acc@0.3m 50.2%

Per-distance bucket (rounded to 0.3 m):
dist   N    Acc@0.3m   MAE    MedAE   P95AE
 2.5   256     28.5%   0.541  0.523  1.184
 5.0   237     67.5%   0.250  0.177  0.721
 6.0   171     58.5%   0.325  0.248  0.891
